In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/tarun/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4


In [14]:
import pandas as pd

from os import listdir
from os.path import isfile, join
files = [f for f in listdir(path) if isfile(join(path, f))]

print(files)

['yelp_academic_dataset_checkin.json', 'Dataset_User_Agreement.pdf', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_business.json', 'yelp_academic_dataset_user.json']


In [15]:
businesses_file = f"{path}/yelp_academic_dataset_business.json"
reviews_file = f"{path}/yelp_academic_dataset_business.json"
users_file = f"{path}/yelp_academic_dataset_user.json"


I want only places that have restaurant as a category, I will convert the json to a csv and then we will import the data into neo4j

I dont need a class just do each one individually

In [16]:
from secret import NEO4J_PASSWORD

from neo4j import GraphDatabase, Driver

URI = "neo4j://localhost"
AUTH = ("neo4j", NEO4J_PASSWORD)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [17]:
def f(tx):
    query = """
        LOAD CSV WITH HEADERS
        FROM "file:///Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/src/business.csv" AS row
        UNWIND row as r
        MERGE (b: Business {
            businessId: r.business_id
        })

        MERGE (c:Category { name: r.categories } )
        MERGE (b)-[:HAS_CATEGORY]->(c)
    """
    try:
        result = tx.run(query)
        return result
    except Exception as e:
        raise e

def create_businesses_and_categories(driver: Driver):
    with driver.session(database="neo4j") as session:
        session.execute_write(
            f
        )
    return

driver = GraphDatabase.driver(URI, auth=AUTH)
try:
    # create_businesses_and_categories(driver)
    pass
except Exception as e:
    raise e
finally:
    driver.close()


In [18]:
import numpy as np

def create_csvs(df: pd.DataFrame):
    df["categories"] = df["categories"].str.split(",")
    cols = ["attributes", "hours"]
    df_ = pd.DataFrame({col: df.pop(col) for col in cols})
    df_["business_id"] = df["business_id"]
    df = df.explode(["categories"])
    df.to_csv(f"../data/business.csv", index=False)
    df_.to_csv(f"../data/business_attributes.csv", index=False)
    return

def restaurant_filter(df: pd.DataFrame) -> pd.DataFrame:
    _df: pd.DataFrame = df[["business_id", "categories"]].copy()
    _df["categories"] = _df["categories"].str.split(",")
    _df = _df.explode(["categories"])
    filtered_ids = _df[_df["categories"] == "Restaurants"]["business_id"]
    return pd.merge(df, filtered_ids, on="business_id", how="inner")

def clean_restuarants_df(df):
    df["postal_code"] = (
    df["postal_code"]
      .astype("string")          
      .replace({"": np.nan, "NULL": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan})
    )

    df.dropna(subset=["business_id", "name", "city", "state", "postal_code"], inplace=True)
    df.drop(columns=["is_open"], inplace=True)
    return df


def create_filtered_restaurant_csvs(df_business):
    df = clean_restuarants_df(restaurant_filter(df_business).copy())
    
    cols = ["attributes", "hours"]
    df_ = pd.DataFrame({col: df.pop(col) for col in cols})
    df_["business_id"] = df["business_id"]

    df_b_categories = df[["business_id", "categories"]].copy()
    
    df_b_categories["categories"] = df_b_categories["categories"].str.split(",")
    df_b_categories = df_b_categories.explode(["categories"]).copy()

    df_categories = pd.DataFrame(df_b_categories["categories"].unique(), columns=["categories"])

    business_categires_mapping = df_categories["categories"].reset_index().set_index("categories")["index"]

    df_categories["categories"] = df_categories["categories"].str.strip()

    df_b_categories["categories"] = df_b_categories["categories"].map(business_categires_mapping)

    df.drop(columns=["categories"], inplace=True)

    df.to_csv(f"../data/business.csv", index=False)
    df_.to_csv(f"../data/business_attributes.csv", index=False)
    df_categories.to_csv(f"../data/categories.csv")
    df_b_categories.to_csv(f"../data/business_categories.csv", index=False)
    return 

In [19]:
create_filtered_restaurant_csvs(df_business=pd.read_json(businesses_file, lines=True))

In [20]:
import matplotlib as plt
# how can we get a list of cuisines, these are very general categories
df = clean_restuarants_df(
    restaurant_filter(pd.read_json(businesses_file, lines=True))
)
x = df[["business_id", "categories"]].copy()
x["categories"] = x["categories"].str.split(",")
x: pd.DataFrame = x.explode(["categories"]).copy()
x["categories"]

0                       Restaurants
0                              Food
0                        Bubble Tea
0                      Coffee & Tea
0                          Bakeries
                    ...            
15289                  Comfort Food
15289                          Food
15289                   Food Trucks
15289                      Caterers
15289     Event Planning & Services
Name: categories, Length: 57472, dtype: object

In [21]:
import psycopg

def run_execute_query(conn, sql: str, params=None, *, fetchone=False, fetchall=False):
    with conn.cursor() as cur:
        cur.execute(sql, params)

        if fetchone:
            return cur.fetchone()
        if fetchall:
            return cur.fetchall()

def copy_from_csv(conn: psycopg.Connection, table, file_path, columns=None, chunk_size=1024 * 1024):
    if columns:
        cols = ", ".join(columns)
        sql = f"COPY {table} ({cols}) FROM STDIN WITH CSV HEADER"
    else:
        sql = f"COPY {table} FROM STDIN WITH CSV HEADER"

    with conn.cursor() as cur:
        with cur.copy(sql) as copy:
            with open(file_path, "r", buffering=chunk_size) as f:
                while True:
                    chunk = f.read(chunk_size)
                    if not chunk:
                        break
                    copy.write(chunk)


In [22]:
create_temp_table_restaurant = """ 
    create temp table staging (
        business_id text,
        name text,
        address text,
        city text,
        state text,
        postal_code text,
        latitude float,
        longitiude float,
        stars float,
        review_count int
    ); """

insert_restaurants_query = """ 
    INSERT INTO restaurants (restaurant_id, name, address, city, state, postal_code, latitude, longitude, stars, review_count)
    SELECT
        business_id,
        name,
        address,
        city,
        state,
        postal_code,
        latitude,
        longitiude,
        stars,
        review_count
    FROM staging;
    """

In [23]:
import psycopg

# Create tables
with psycopg.connect("dbname = beeg") as conn:
    with open(r"create.sql") as file:
        create_sql_query = file.read()
        run_execute_query(conn, create_sql_query)

# Populate tables
with psycopg.connect("dbname = beeg") as conn:
    run_execute_query(conn, create_temp_table_restaurant)
    copy_from_csv(conn, "staging", "/Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/data/business.csv")
    run_execute_query(conn, insert_restaurants_query)

In [24]:
# Populate tables
with psycopg.connect("dbname = beeg") as conn:
    copy_from_csv(conn, "categories", "/Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/data/categories.csv")

In [25]:
with psycopg.connect("dbname = beeg") as conn:
    copy_from_csv(conn, "restaurant_categories", "/Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/data/business_categories.csv")

In [26]:
import polars as pl

# using polars as the amount of data here is quite large
df = pl.read_ndjson(users_file)
df

user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
str,str,i64,str,i64,i64,i64,str,str,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""qVc8ODYU5SZjKXVBgXdI7w""","""Walker""",585,"""2007-01-25 16:47:26""",7217,1259,5994,"""2007""","""NSCy54eWehBJyZdG2iE84w, pe42u7…",267,3.91,250,65,55,56,18,232,844,467,467,239,180
"""j14WgRoU_-2ZE1aw1dXrJg""","""Daniel""",4333,"""2009-01-25 04:35:42""",43091,13066,27281,"""2009,2010,2011,2012,2013,2014,…","""ueRPE0CX75ePGMqOFVj6IQ, 52oH4D…",3138,3.74,1145,264,184,157,251,1847,7054,3131,3131,1521,1946
"""2WnXYQFK0hXEoTxPtV2zvg""","""Steph""",665,"""2008-07-25 10:41:00""",2086,1010,1003,"""2009,2010,2011,2012,2013""","""LuO3Bn4f3rlhyHIaNfTlnA, j9B4Xd…",52,3.32,89,13,10,17,3,66,96,119,119,35,18
"""SZDeASXq7o05mMNLshsdIA""","""Gwen""",224,"""2005-11-29 04:38:33""",512,330,299,"""2009,2010,2011""","""enx1vVPnfdNUdPho6PH_wg, 4wOcvM…",28,4.27,24,4,1,6,2,12,16,26,26,10,9
"""hA5lMy-EnncsH4JoR-hFGQ""","""Karen""",79,"""2007-01-05 19:40:59""",29,15,7,"""""","""PBK4q9KEEBHhFvSXCUirIw, 3FWPpM…",1,3.54,1,1,0,0,0,1,1,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""fB3jbHi3m0L2KgGOxBv6uw""","""Jerrold""",23,"""2015-01-06 00:31:31""",7,0,0,"""""","""None""",0,4.92,0,0,0,0,0,0,0,0,0,0,0
"""68czcr4BxJyMQ9cJBm6C7Q""","""Jane""",1,"""2016-06-14 07:20:52""",0,0,0,"""""","""None""",0,5.0,0,0,0,0,0,0,0,0,0,0,0
"""1x3KMskYxOuJCjRz70xOqQ""","""Shomari""",4,"""2017-02-04 15:31:58""",1,1,0,"""""","""None""",0,2.0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
class DuplicateError(Exception):
    pass

def make_users_csv(df):
    df = df.with_columns(
        pl.concat_str(
            [
                pl.col("name"), 
                pl.col("user_id").str.tail(8),
            ],
            separator="_"
        ).alias("username")
    )
    if df["username"].is_duplicated().any():
        raise DuplicateError("duplicate usernames exists")

    filtered_df = df.select(["user_id", "username"])
    filtered_df.write_csv(file="../data/users.csv")
    return

def make_users_following_csv(df: pl.DataFrame):
    (
        df
        .select(["user_id", "friends"])
        .lazy()
        .with_columns(
            pl.col("friends")
            .str.split(",")
            .alias("friends_split")
        )
        
        .drop("friends")
        .explode("friends_split")
        .with_columns(
            pl.col("friends_split")
            .replace(
                {
                    "": None,
                    "NULL": None,
                    "None": None,
                    "nan": None,
                    "NaN": None,
                }
            )
            .str.strip()
        )
        .drop_nulls()
        .sink_csv("../data/user_following.csv")
    )
    return

In [28]:
make_users_csv(df)

In [29]:
with psycopg.connect("dbname = beeg") as conn:
    copy_from_csv(conn, "users", "/Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/data/users.csv", columns=("user_id", "username"))

In [34]:
make_users_following_csv(df)

AttributeError: 'LazyFrame' object has no attribute 'str'

In [33]:
with psycopg.connect("dbname = beeg") as conn:
    copy_from_csv(conn, "user_following", "/Users/tarun/Prog/assignment2-at2025-snow-flake-2512-main/beeg/data/user_following.csv")

ForeignKeyViolation: insert or update on table "user_following" violates foreign key constraint "user_following_follows_id_fkey"
DETAIL:  Key (follows_id)=( pe42u7DcCH2QmI81NX-8qA) is not present in table "users".